# 🎬 ActionNet Training — GRU Action Recognition

**Kiến trúc**: MediaPipe Pose keypoints → GRU(128) → 8 action classes  
**Input** : (batch, 30, 132) — 30 frames × 33 keypoints × 4 features  
**Output** : 8 actions: standing, walking, running, falling, climbing, fighting, raising_hand, gathering

## Yêu cầu
Upload lên Google Drive:
- `actionnet_keypoints.zip` — thư mục `data/actionnet_dataset/keypoints/`

In [ ]:
# ============================================================
# 1. Kiểm tra runtime
# ============================================================
import torch
import subprocess

print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ============================================================
# 2. Mount Drive & Unzip data
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

ZIP_PATH = '/content/drive/MyDrive/actionnet_keypoints.zip'
DATA_DIR = '/content/actionnet_data'

if os.path.exists(ZIP_PATH):
    print(f'Extracting {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('Done!')
else:
    print(f'[WARN] {ZIP_PATH} not found. Using demo data...')
    DATA_DIR = None

KEYPOINTS_DIR = os.path.join(DATA_DIR or '', 'keypoints') if DATA_DIR else None
print(f'Keypoints dir: {KEYPOINTS_DIR}')
if KEYPOINTS_DIR and os.path.exists(KEYPOINTS_DIR):
    csvs = [f for f in os.listdir(KEYPOINTS_DIR) if f.endswith('.csv')]
    print(f'Total CSV files: {len(csvs)}')

In [ ]:
# ============================================================
# 3. Dataset Loading
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

ACTION_CLASSES = [
    'standing', 'walking', 'running', 'falling',
    'climbing', 'fighting', 'raising_hand', 'gathering'
]
ACTION2IDX = {a: i for i, a in enumerate(ACTION_CLASSES)}

WINDOW_FRAMES = 30
INPUT_DIM     = 132


def load_keypoints_dataset(keypoints_dir):
    """Load tất cả CSV files thành X (N, 30, 132) và y (N,)."""
    X_list, y_list = [], []
    kp_dir = Path(keypoints_dir)

    for csv_path in sorted(kp_dir.glob('*.csv')):
        # Tên file: {action}_{idx}.csv
        action = csv_path.stem.rsplit('_', 1)[0]
        if action not in ACTION2IDX:
            continue

        df = pd.read_csv(csv_path)
        # Lấy cột keypoint (bỏ cột action ở cuối nếu có)
        kp_cols = [c for c in df.columns if c.startswith('kp_')]
        if len(df) < WINDOW_FRAMES:
            continue

        kp = df[kp_cols].values[:WINDOW_FRAMES].astype(np.float32)  # (30, 132)
        X_list.append(kp)
        y_list.append(ACTION2IDX[action])

    X = np.array(X_list)   # (N, 30, 132)
    y = np.array(y_list)   # (N,)
    return X, y


def generate_demo_data(n_per_class=100):
    """Sinh dữ liệu demo ngẫu nhiên để test pipeline khi không có data thật."""
    X_list, y_list = [], []
    for idx, action in enumerate(ACTION_CLASSES):
        for _ in range(n_per_class):
            # Keypoints ngẫu nhiên với một chút pattern
            base = np.random.randn(WINDOW_FRAMES, INPUT_DIM).astype(np.float32)
            # Thêm shift nhỏ theo action idx để model có thể phân biệt
            base[:, idx * 16 : idx * 16 + 16] += idx * 0.5
            X_list.append(base)
            y_list.append(idx)
    return np.array(X_list), np.array(y_list)


# Load data
if KEYPOINTS_DIR and os.path.exists(KEYPOINTS_DIR):
    print('Loading real keypoints data...')
    X, y = load_keypoints_dataset(KEYPOINTS_DIR)
    USE_REAL_DATA = True
else:
    print('[WARN] Không có data thật. Dùng demo data để test pipeline.')
    X, y = generate_demo_data(n_per_class=200)
    USE_REAL_DATA = False

print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'Class distribution: {dict(Counter(y))}')
labels_dist = {ACTION_CLASSES[k]: v for k, v in Counter(y).items()}
print(f'Action dist: {labels_dist}')

In [ ]:
# ============================================================
# 4. Data Normalization & Split
# ============================================================
from sklearn.preprocessing import StandardScaler

# Normalize: (N, 30, 132) → flatten → scale → reshape
N = len(X)
X_flat = X.reshape(N, -1)   # (N, 30*132)
scaler  = StandardScaler()
X_flat  = scaler.fit_transform(X_flat)
X       = X_flat.reshape(N, WINDOW_FRAMES, INPUT_DIM).astype(np.float32)

# Train/Val/Test split (80/10/10)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# Tính class weights cho WeightedRandomSampler
class_counts = Counter(y_train)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = torch.tensor([class_weights[yi] for yi in y_train], dtype=torch.float)

print(f'Class weights: {class_weights}')

In [ ]:
# ============================================================
# 5. Dataset & DataLoader
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

def make_loader(X, y, batch_size=64, sampler=None, shuffle=True):
    ds = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).long()
    )
    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=(shuffle and sampler is None),
        num_workers=2,
        pin_memory=True,
    )

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

BATCH_SIZE = 64
train_loader = make_loader(X_train, y_train, BATCH_SIZE, sampler=weighted_sampler)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# ============================================================
# 6. Model Architecture — ActionNet GRU
# ============================================================
class ActionNetGRU(nn.Module):
    """
    GRU-based action recognizer.
    Input : (batch, 30, 132)  — sequence of keypoint frames
    Output: (batch, 8)        — action class logits
    """
    def __init__(self, input_dim=132, hidden_size=128, num_layers=2,
                 num_classes=8, dropout=0.4):
        super().__init__()

        self.gru = nn.GRU(
            input_size  = input_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
            bidirectional = False,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        """x: (batch, seq_len, input_dim)"""
        out, _ = self.gru(x)           # out: (batch, seq, hidden)
        return self.head(out[:, -1, :]) # lấy hidden state cuối


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

model = ActionNetGRU(
    input_dim   = INPUT_DIM,
    hidden_size = 128,
    num_layers  = 2,
    num_classes = len(ACTION_CLASSES),
    dropout     = 0.4,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

# Test forward
dummy = torch.randn(4, WINDOW_FRAMES, INPUT_DIM).to(DEVICE)
out   = model(dummy)
print(f'Output shape: {out.shape}')  # should be (4, 8)

In [ ]:
# ============================================================
# 7. Training Setup
# ============================================================
from torch.optim.lr_scheduler import OneCycleLR

NUM_EPOCHS   = 50
LR           = 3e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr         = LR,
    steps_per_epoch= len(train_loader),
    epochs         = NUM_EPOCHS,
    pct_start      = 0.1,  # 10% warmup
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Optimizer: AdamW(lr={LR}, wd={WEIGHT_DECAY})')
print(f'Scheduler: OneCycleLR({NUM_EPOCHS} epochs)')
print(f'Criterion: CrossEntropyLoss(label_smoothing=0.1)')

In [ ]:
# ============================================================
# 8. Training Loop
# ============================================================
import time

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total


# Train!
best_val_acc = 0.0
best_model_state = None
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Training {NUM_EPOCHS} epochs on {DEVICE}...')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"Time":>5}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE)

    elapsed = time.time() - t0
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    marker = ' *' if vl_acc > best_val_acc else ''
    if vl_acc > best_val_acc:
        best_val_acc   = vl_acc
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch <= 5:
        print(f'{epoch:>5} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | '
              f'{vl_loss:>8.4f} | {vl_acc*100:>6.2f}%{marker} | {elapsed:>4.0f}s')

print(f'\nBest Val Acc: {best_val_acc*100:.2f}%')

In [ ]:
# ============================================================
# 9. Đánh giá trên Test Set
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(DEVICE)
        logits = model(X_b)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_b.numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Test Accuracy: {test_acc*100:.2f}%')
print()
print(classification_report(all_labels, all_preds, target_names=ACTION_CLASSES))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ACTION_CLASSES, yticklabels=ACTION_CLASSES, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title(f'ActionNet Confusion Matrix (Test Acc: {test_acc*100:.1f}%)')
plt.tight_layout()
plt.savefig('/content/actionnet_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 10. Learning Curve
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history['train_acc']], label='Train Acc')
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   label='Val Acc')
ax2.axhline(y=80, color='r', linestyle='--', alpha=0.5, label='Target 80%')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Training Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/actionnet_learning_curve.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 11. Export Model
# ============================================================
import json
from datetime import datetime

# Lưu checkpoint
checkpoint = {
    'model_state' : best_model_state,
    'val_acc'     : best_val_acc,
    'test_acc'    : test_acc,
    'action_classes': ACTION_CLASSES,
    'input_dim'   : INPUT_DIM,
    'window_frames': WINDOW_FRAMES,
    'hidden_size' : 128,
    'num_layers'  : 2,
    'use_real_data': USE_REAL_DATA,
    'trained_at'  : datetime.now().isoformat(),
}
torch.save(checkpoint, '/content/actionnet_gru.pt')
print(f'Saved: actionnet_gru.pt')

# Metadata JSON
metadata = {
    'model'         : 'ActionNet-GRU',
    'version'       : 'v1',
    'architecture'  : 'GRU(hidden=128, layers=2) → FC(64) → 8 classes',
    'input_shape'   : [WINDOW_FRAMES, INPUT_DIM],
    'num_classes'   : len(ACTION_CLASSES),
    'action_classes': ACTION_CLASSES,
    'best_val_acc'  : round(best_val_acc * 100, 2),
    'test_acc'      : round(test_acc * 100, 2),
    'use_real_data' : USE_REAL_DATA,
    'trained_at'    : datetime.now().isoformat(),
}
with open('/content/actionnet_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved: actionnet_metadata.json')
print(f'\n=== RESULTS ===')
print(f'Best Val Accuracy: {best_val_acc*100:.2f}%')
print(f'Test Accuracy    : {test_acc*100:.2f}%')

In [ ]:
# ============================================================
# 12. Copy lên Drive & Download
# ============================================================
import shutil

SAVE_DIR = '/content/drive/MyDrive/actionnet_output'
os.makedirs(SAVE_DIR, exist_ok=True)

for fname in ['actionnet_gru.pt', 'actionnet_metadata.json',
              'actionnet_confusion_matrix.png', 'actionnet_learning_curve.png']:
    src = f'/content/{fname}'
    dst = f'{SAVE_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved to Drive: {dst}')

print('\n=== Sau khi train xong ===')
print('Copy 2 files về máy:')
print('  actionnet_gru.pt')
print('  actionnet_metadata.json')
print('Paste vào: c:\\code\\camera-ai\\models\\')
print('Chạy test: python -X utf8 tests/test_actionnet.py')